Importing necessary libraries

In [1]:

# Data handling library
import pandas as pd

# Machine learning libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


Making Preprocessing Class

In [2]:
# Preprocessing steps
data = pd.read_csv('Dataset/Titanic-Dataset.csv')


class Preprocessing:
    def __init__(self):
        pass

    def feature_engineering(self, data):
        # Dropping unnecessary columns
        input_data = data.drop(columns=['PassengerId', 'Survived', 'Ticket', 'Cabin'])

        # Creating target variable
        output_data = data['Survived']

        # Feature engineering: creating  'Family_size'
        input_data['Family_size'] = input_data['Parch'] + input_data['SibSp'] + 1

        # Dropping columns used for feature creation
        input_data = input_data.drop(columns=['Name', 'SibSp', 'Parch'])

        return input_data, output_data

    def splitting(self, input_data, output_data):
        # Splitting before preprocessing to avoid data leakage
        X_train, X_test, y_train, y_test = train_test_split(
            input_data, output_data, test_size=0.2, random_state=42
        )
        return X_train, X_test, y_train, y_test

    def handle_null_values(self, X_train, X_test):
        # Filling missing values in 'Embarked' with mode ('S')
        X_train['Embarked'] = X_train['Embarked'].fillna('S')
        X_test['Embarked'] = X_test['Embarked'].fillna('S')

        # Custom imputer for 'Age'
        class AgeImputer:
            def __init__(self):
                self.age_medians = None
                self.global_median = None

            def fit(self, X_train):
                self.age_medians = X_train.groupby(['Sex', 'Pclass'])['Age'].median()
                self.global_median = X_train['Age'].median()
                return self

            def transform(self, X):
                X_copy = X.copy()

                def fill_age(row):
                    if pd.isna(row['Age']):
                        key = (row['Sex'], row['Pclass'])
                        return self.age_medians.get(key, self.global_median)
                    return row['Age']

                X_copy['Age'] = X_copy.apply(fill_age, axis=1)
                X_copy['Age'] = X_copy['Age'].fillna(self.global_median)

                return X_copy

        age_imputer = AgeImputer()
        age_imputer.fit(X_train)

        X_train = age_imputer.transform(X_train)
        X_test = age_imputer.transform(X_test)

        return X_train, X_test

    def one_hot_encoding(self, X_train, X_test):
        # One-hot encoding for categorical variables
        X_train = pd.get_dummies(X_train, drop_first=True)
        X_test = pd.get_dummies(X_test, drop_first=True)

        # Aligning train and test sets to ensure same columns
        X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

        return X_train, X_test

    def final_check_point(self, X_train, X_test):
        # Final validation for missing values and column alignment
        print("Null values in the training set:\n")
        print(X_train.isna().sum())

        print("\nNull values in the test set:\n")
        print(X_test.isna().sum())

        print(f"\nAll columns are aligned correctly: {all(X_train.columns == X_test.columns)}")

        # All preprocessing steps completed successfully

Applying Preprocessing

In [3]:
# Initializing preprocessing class
preprocessing = Preprocessing()
# Feature engineering
input_data, output_data = preprocessing.feature_engineering(data)
# Splitting the data
X_train, x_test, Y_train, y_test = preprocessing.splitting(input_data, output_data)
# Handling null values
X_train, x_test = preprocessing.handle_null_values(X_train, x_test)
# One-hot encoding
X_train, x_test = preprocessing.one_hot_encoding(X_train, x_test)
# Final checkpoint
preprocessing.final_check_point(X_train, x_test)

Null values in the training set:

Pclass         0
Age            0
Fare           0
Family_size    0
Sex_male       0
Embarked_Q     0
Embarked_S     0
dtype: int64

Null values in the test set:

Pclass         0
Age            0
Fare           0
Family_size    0
Sex_male       0
Embarked_Q     0
Embarked_S     0
dtype: int64

All columns are aligned correctly: True


Training a Base line model , Hence here we are making a classification model , So we are building Logistic Regression model as our Baseline Model

In [4]:
# Building a baseline model (Logistic Regression for classification)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(x_test)
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_scaled, Y_train)
# Predictions on test set
predictions_test_logistic = logistic_model.predict(x_test_scaled)
# Evaluation metrics
acc_1 = accuracy_score(y_test, predictions_test_logistic)
recall_1 = classification_report(y_test, predictions_test_logistic, output_dict=True)['1']['recall']
print(f"Accuracy of the Logistic Regression model: {acc_1}")
print(f"\nClassification Report:\n{classification_report(y_test, predictions_test_logistic)}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, predictions_test_logistic)}")

Accuracy of the Logistic Regression model: 0.8212290502793296

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.88      0.85       105
           1       0.81      0.74      0.77        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179


Confusion Matrix:
[[92 13]
 [19 55]]


Building Random Forest Model

In [5]:

param_grid = {
    'n_estimators':[100, 150, 200, 250],
    'max_depth':[10, 15, 20, 25],
    'min_samples_split':[2, 5, 7],
    'min_samples_leaf':[1, 5, 7, 9],
    'max_features':['sqrt', 'log2']
}
grid = GridSearchCV(RandomForestClassifier(class_weight='balanced'), param_grid, cv=5)
grid.fit(X_train, Y_train)
print(f'The best parameters we got using Random Forest Algorithm {grid.best_params_}' )
#The model with the best parameters
best_model = grid.best_estimator_

The best parameters we got using Random Forest Algorithm {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 7, 'n_estimators': 100}


Evaluation Metrics of Random Forest Model

In [6]:
predictions_test_random_forest = best_model.predict(x_test)
#Metrics Accuracy Score , Classification report and Confusion matrix
acc_2 = accuracy_score(y_test, predictions_test_random_forest)
recall_2 = classification_report(y_test,predictions_test_random_forest, output_dict=True)['1']['recall']
print(f'The accuracy of the Random Forest model is {accuracy_score(y_test, predictions_test_random_forest)}')
print(f'The Classification report of the Random Forest is {classification_report(y_test,predictions_test_random_forest)}')
print(f'The Confusion Matrix of the Random Forest is {confusion_matrix(y_test, predictions_test_random_forest)}')


The accuracy of the Random Forest model is 0.8435754189944135
The Classification report of the Random Forest is               precision    recall  f1-score   support

           0       0.86      0.88      0.87       105
           1       0.82      0.80      0.81        74

    accuracy                           0.84       179
   macro avg       0.84      0.84      0.84       179
weighted avg       0.84      0.84      0.84       179

The Confusion Matrix of the Random Forest is [[92 13]
 [15 59]]


XGBoost Model Building

In [7]:
#XGBoost Model Building
param_grid_Xg = {
    'n_estimators':[100, 150, 200, 250],
    'max_depth':[10, 15, 20, 25],
    'learning_rate':[0.1, 0.3, 0.2],
    'subsample':[0.7, 0.8, 0.9]
}
grid_xg = GridSearchCV(XGBClassifier(), param_grid=param_grid_Xg, cv=5)
grid_xg.fit(X_train, Y_train)
print(f'The best parameters we got using XGBoost Algorithm {grid_xg.best_params_}' )
best_model_xg = grid_xg.best_estimator_

The best parameters we got using XGBoost Algorithm {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 100, 'subsample': 0.7}


Evaluation Metrics of XGBOOST Model

In [8]:
predictions_test_xg = best_model_xg.predict(x_test)
#Metrics Accuracy Score , Classification report and Confusion matrix
acc_3 = accuracy_score(y_test, predictions_test_xg)
recall_3 = classification_report(y_test,predictions_test_xg, output_dict=True)['1']['recall']
print(f'The accuracy of the XGBOOST model is {accuracy_score(y_test, predictions_test_xg)}')
print(f'The Classification report of the XGBOOST is {classification_report(y_test,predictions_test_xg)}')
print(f'The Confusion Matrix of the XGBOOST is {confusion_matrix(y_test, predictions_test_xg)}')


The accuracy of the XGBOOST model is 0.8212290502793296
The Classification report of the XGBOOST is               precision    recall  f1-score   support

           0       0.84      0.86      0.85       105
           1       0.79      0.77      0.78        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179

The Confusion Matrix of the XGBOOST is [[90 15]
 [17 57]]


In [9]:
#comparison table
results = {
    'Model':['Logistic Model', 'Random Forest', 'XGBOOST'],
    'Accuracy':[acc_1, acc_2, acc_3],
    'Recall':[recall_1, recall_2, recall_3]
}

comparison_table = pd.DataFrame(results)
print(comparison_table)


            Model  Accuracy    Recall
0  Logistic Model  0.821229  0.743243
1   Random Forest  0.843575  0.797297
2         XGBOOST  0.821229  0.770270


#By we can conclude here the best performing model is XGBOOST as it has more recall  but the difference between Random Forest model and XGBOOST is very little , while XGBOOST is a complex and more sensitive than random forest , so we would choose random forest model as the final model out of all 3 and Random forest is the best suited model for the following dataset